# Assignment 6

## Importing the data
importing the data and assigning the features, the target, and the images themselves to seperate variables
I am also turning the data into a dataframe as the returned data from the import is only a numpy array

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_olivetti_data

data = fetch_olivetti_data(shuffle=True, random_state=42)

## one column per pixel
pixel_cols = [f"pixel_{i}" for i in range(data.data.shape[1])]

df = pd.DataFrame(data.data, columns=pixel_cols)
df["target"] = data.target

print(df.shape)

In [ ]:
df.head(20)

## data inspection
printing the amount of samples, how many unique samples there are, the dimensionality of the features and the amount of images per subject

In [ ]:
print("Samples:", len(df))
print("Unique subjects:", df["target"].nunique())
print("Feature dimensions:", df.shape[1] - 1)
print(df["target"].value_counts().sort_index().head())  # 10 images per subject

## assign data to seperate arrays

I am doing this because I don't want to be using a dataframe for the actual modeling

In [ ]:
X = data.data
y = data.target

## plotting the subject images
Here, I am plotting the images. 
I wanted to make sure this works independant of whether you import the data using the shuffled=True call or not
To do this, I just created an array with the hardcoded ID's I wanted to use. 
Then, I found and image for each ID (ID's are stored in target array)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# One image from each of the first 20 subjects (0–19)
subject_ids = np.arange(20)
idx = [np.where(data.target == s)[0][0] for s in subject_ids]

fig, axes = plt.subplots(4, 5, figsize=(10, 8.5))

for ax, i in zip(axes.ravel(), idx):
    ax.imshow(data.images[i], cmap="gray")
    ax.set_title(f"Subject {data.target[i]}", fontsize=10)
    ax.axis("off")

fig.suptitle("Olivetti Faces: Sample Images from 20 Subjects", fontsize=14)
plt.show()

## Plotting cume explained variance and Elbow Plot

Here, I am fitting a full PCA model to all 400 vectors to see how the dtaset's total variance is spread across each component

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

X = data.data
y = data.target

## Fit a full PCA (no truncation). With 400 samples, at most 400 components exist.
pca_full = PCA()
pca_full.fit(X)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

print(f"Original feature dimensions: {X.shape[1]}")
print(f"Principal components computed: {len(explained)}")
print(f"Variance explained by PC1: {explained[0]:.4f}")
print(f"Variance explained by first 10 PCs: {cumulative[9]:.4f}\n")

## Smallest number of components that reaches each threshold
thresholds = [0.90, 0.95, 0.99]
n_components = {}
for t in thresholds:
    k = np.argmax(cumulative >= t) + 1      ## +1 because indices start at 0
    n_components[t] = k
    print(f"Components needed for {int(t*100)}% variance: {k}  "
          f"({k / X.shape[1] * 100:.2f}% of the original 4,096 dimensions)")

n_components_95 = n_components[0.95]        ## This will be reused in tasks 5-8

## The actual plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
pcs = np.arange(1, len(explained) + 1)

## Left: individual explained variance ratio (elbow plot)
ax1.plot(pcs, explained, marker=".", color="tab:blue")
ax1.set_xlim(0, 60)                          ## zoom in on the early components where the elbow is
ax1.set_title("Individual Explained Variance Ratio (Elbow Plot)")
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance Ratio")
ax1.grid(True, alpha=0.3)

## Right: cumulative explained variance with the three thresholds
ax2.plot(pcs, cumulative, color="tab:blue", linewidth=2)
colors = {0.90: "tab:green", 0.95: "tab:orange", 0.99: "tab:red"}
for t, k in n_components.items():
    ax2.axhline(t, color=colors[t], linestyle="--", linewidth=1)
    ax2.axvline(k, color=colors[t], linestyle="--", linewidth=1,
                label=f"{int(t*100)}% → {k} components")
ax2.set_title("Cumulative Explained Variance")
ax2.set_xlabel("Number of Principal Components")
ax2.set_ylabel("Cumulative Explained Variance Ratio")
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Above we see that 90% = 66 components, 95% = 123 components and 99% = 260 components. Telling us the rest of the 160 components are basically just background noise
